# 🚀 GNOT RF Cavity — Google Colab Eğitim Pipeline (Tek GPU)

Bu notebook, GNOT modelini Google Colab'de **Tek GPU** ile eğitmek için hazırlanmıştır.

## ⚡ Önemli Notlar
- **Runtime (Çalışma Zamanı)**: T4, L4 veya A100 GPU seçili olmalıdır (Runtime → Change runtime type)
- **Veri Formatı**: Hızlı yükleme ve RAM tasarrufu için `.h5` kullanılmaktadır.
- **Config**: `configs/default.yaml` (Tek GPU için optimize edilmiştir)

## 1. 📁 Google Drive Bağlantısı (İsteğe Bağlı)
Dataset ve checkpoint'leri kalıcı olarak saklamak için Drive'ı bağlayabilirsiniz.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive bağlandı!")
except ImportError:
    print("Colab ortamında değilsiniz veya Drive bağlanmadı.")

## 2. 🔧 Kurulum ve Repo Klonlama

In [ ]:
from getpass import getpass
import os

# GitHub Token
token = getpass('GitHub Personal Access Token (PAT) giriniz: ')

# Repo klonla (/content altında)
%cd /content
repo_url = f"https://{token}@github.com/KorayGokceler/rf_cavity_neural_operator.git"
!git clone {repo_url}
%cd rf_cavity_neural_operator

# Sistem bağımlılıkları (gmsh için gerekli grafik kütüphaneleri)
!apt-get install -y libglu1-mesa libxcursor1 libxinerama1 libxft2 libxrender1 --quiet

# Python kütüphaneleri
!pip install -r requirements.txt

In [ ]:
# Yeni güncellemeleri çekmek isterseniz:
!git pull origin main
print("✅ Güncellemeler alındı!")

## 3. 🖥️ GPU Doğrulama

In [ ]:
import torch

n_gpus = torch.cuda.device_count()
print(f"\n{'='*50}")
print(f"  GPU Sayısı: {n_gpus}")
print(f"{'='*50}")

if n_gpus > 0:
    props = torch.cuda.get_device_properties(0)
    print(f"  Aktif GPU: {props.name}")
    print(f"  VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"  Compute Capability: {props.major}.{props.minor}")
else:
    print("\n❌ GPU bulunamadı! Runtime -> Change runtime type alanından Hardware Accelerator olarak GPU seçin.")

## 4. 📦 Veri Seti Yükleme/Üretme (.h5 Formatında)

In [ ]:
import os

os.makedirs("data", exist_ok=True)

# === Drive'da daha önceden üretilmiş dataset varsa kopyala (İsteğe bağlı) ===
# drive_dataset_path = "/content/drive/MyDrive/gnot_dataset.h5"
# if os.path.exists(drive_dataset_path):
#     !cp {drive_dataset_path} data/gnot_dataset.h5
#     print("✅ Dataset Drive'dan kopyalandı!")

if not os.path.exists("data/gnot_dataset.h5"):
    print("Dataset bulunamadı, sıfırdan üretiliyor... (Bu işlem birkaç dakika sürebilir)")
    
    # 1) H5 dataset üret (1000 sample)
    !python src/data_gen/dataset_generator.py --h5_filename rf_cavity_1000_dataset.h5 --n_total 1000 --n_plot 100 --mode random
    
    # 2) GNOT H5 formatına dönüştür
    !python convert.py --h5_filepath rf_cavity_1000_dataset.h5 --output_path data/gnot_dataset.h5 --output_format h5 --modes 0 1 2
    
    # Drive'a yedekle (Drive bağlıysa ve isterseniz)
    # !cp data/gnot_dataset.h5 /content/drive/MyDrive/
else:
    print("✅ Dataset mevcut!")

# Boyut kontrolü
if os.path.exists("data/gnot_dataset.h5"):
    size_mb = os.path.getsize("data/gnot_dataset.h5") / 1e6
    print(f"\n📦 Dataset formatı hazır. Boyut: {size_mb:.1f} MB")

## 5. 🏋️ Eğitim — Tek GPU (default.yaml)

In [ ]:
# Hızlı test (pipeline çalışıyor mu diye 1 epoch)
# !python train.py --config configs/default.yaml --fast_dev_run

In [ ]:
# Tam Eğitim (default: batch_size=16, epochs=50)
# Colab T4 (15GB VRAM) için batch=16 ve gradient checkpointing (~7-8GB kullanır)
!python train.py --config configs/default.yaml

## 6. 📊 TensorBoard İzleme

In [ ]:
%load_ext tensorboard
%tensorboard --logdir training_logs

## 7. 📥 Sonuçları ve Modeli Yedekleme

In [ ]:
import glob

# En iyi checkpoint'ı bul
ckpts = glob.glob("training_logs/gnot_run/**/best-*.ckpt", recursive=True)
if ckpts:
    best_ckpt = sorted(ckpts, key=os.path.getmtime)[-1]
    print(f"✅ En iyi checkpoint: {best_ckpt}")
    
    # Drive'a kopyala
    # !cp {best_ckpt} /content/drive/MyDrive/best_model.ckpt
    # print("Checkpoint Drive'a kaydedildi.")
else:
    print("❌ Checkpoint bulunamadı!")